# 🏠 House Price Prediction — V11 (XGB + LGB Ensemble + Optuna)
**New features**: 17 NLP features extracted from listing titles and descriptions
**Pipeline**: Load from BigQuery → Dedup → Outlier removal → Feature engineering → Optuna tuning → Ensemble

## 1. Install packages

In [1]:
!pip install xgboost lightgbm category_encoders underthesea optuna google-cloud-bigquery -qq

## 2. Imports

In [2]:
import pandas as pd
import numpy as np
import joblib
import optuna
import xgboost as xgb
import lightgbm as lgb
import category_encoders as ce
from google.cloud import bigquery
from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
from underthesea import text_normalize

optuna.logging.set_verbosity(optuna.logging.WARNING)
print('All packages imported successfully ✅')

## 3. Load data from BigQuery

In [19]:
# ── Cấu hình ──────────────────────────────────────────────────────────────────
GCP_KEY_PATH  = '/lakehouse/default/Files/real-estate-project-445516-83dc50b692bc.json'
BQ_TABLE      = 'real-estate-project-445516.real_estate_data.ads_data'
LAKEHOUSE_DIR = '/lakehouse/default/Files/training_data'
# ──────────────────────────────────────────────────────────────────────────────

client = bigquery.Client.from_service_account_json(GCP_KEY_PATH)

query = f"""
    SELECT *
    FROM `{BQ_TABLE}`
    WHERE `Ngày gia hạn` >= '2026-03-01T00:00:00'
      AND `Ngày gia hạn` <= '2026-03-31T00:00:00'
"""

# df = client.query(query).to_dataframe()
# Hoặc load từ file CSV đã có sẵn:
df = pd.read_csv(f'{LAKEHOUSE_DIR}/ads_data_2026_03.csv', low_memory=False)

print(f'Loaded {len(df):,} rows, {df.shape[1]} columns')
df.head(2)

## 4. Preprocessing & Deduplication

In [20]:
TOP_4_TYPES = ['căn hộ chung cư', 'nhà riêng', 'đất', 'nhà mặt phố']

df = df[
    (df['Loại quảng cáo'] == 'Bán') &
    (df['Loại BĐS'].isin(TOP_4_TYPES)) &
    (df['Tỉnh thành phố'] == 'Hà Nội')
].copy()

for col in ['Khoảng giá', 'Diện tích', 'Tọa độ x', 'Tọa độ y']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df[df['Khoảng giá'] > 1e8].dropna(subset=['Khoảng giá', 'Diện tích'])
print(f'After initial filter: {len(df):,} rows')

# ── Deduplication theo 15 cột cấu trúc ────────────────────────────────────────
GROUP_COLS = [
    'Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố', 'Quận', 'Địa chỉ 1', 'Căn góc',
    'Diện tích', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Tên dự án',
    'Hướng nhà', 'Hướng ban công', 'Số tầng', 'Mặt tiền', 'Đường vào',
]

df_dedup = (
    df.groupby(GROUP_COLS, dropna=False)
    .agg(
        Price=('Khoảng giá', lambda s: np.mean(np.unique(s[s.notna()])) if s.notna().any() else np.nan),
        Pháp_lý=('Pháp lý', 'first'),
        Nội_thất=('Nội thất', 'first'),
        Địa_chỉ_2=('Địa chỉ 2', 'first'),
        Tiêu_đề=('Tiêu đề', 'first'),
        Mô_tả=('Mô tả', 'first'),
        Phường=('Phường Xã Thị trấn', 'first'),
        Tọa_độ_x=('Tọa độ x', 'first'),
        Tọa_độ_y=('Tọa độ y', 'first'),
    )
    .reset_index()
    .rename(columns={
        'Pháp_lý': 'Pháp lý', 'Nội_thất': 'Nội thất', 'Địa_chỉ_2': 'Địa chỉ 2',
        'Tiêu_đề': 'Tiêu đề', 'Mô_tả': 'Mô tả', 'Phường': 'Phường Xã Thị trấn',
        'Tọa_độ_x': 'Tọa độ x', 'Tọa_độ_y': 'Tọa độ y',
    })
    .dropna(subset=['Price'])
)
print(f'After dedup: {len(df_dedup):,} rows')

# ── Drop top 1% giá cao nhất theo (Loại BĐS, Quận) ───────────────────────────
idx_drop = []
for _, g in df_dedup.groupby(['Loại BĐS', 'Quận']):
    k = max(1, int(np.ceil(len(g) * 0.01)))
    idx_drop.extend(g.nlargest(k, 'Price').index.tolist())

df_final = df_dedup.drop(index=idx_drop).reset_index(drop=True)
print(f'After top-1% drop: {len(df_final):,} rows')

In [21]:
df_final.info()

## 5. Feature Engineering

In [ ]:
METRO_STATIONS = [
    (21.028, 105.828), (21.015, 105.820), (21.015, 105.810),
    (21.030, 105.800), (21.002, 105.815), (20.975, 105.776),
]
CENTER_LAT, CENTER_LON = 21.0285, 105.8542

def extract_features(df):
    print('Normalizing text and extracting features...')
    df = df.copy()
    df['clean_desc'] = df['Mô tả'].astype(str).apply(text_normalize).str.lower()
    desc = df['clean_desc']

    # ── Search cả tiêu đề và mô tả ───────────────────────────────────────────
    if 'Tiêu đề' in df.columns:
        df['clean_title'] = df['Tiêu đề'].astype(str).apply(text_normalize).str.lower()
        text = df['clean_title'] + ' ' + desc  # kết hợp cả title và desc để search
    else:
        df['clean_title'] = ''
        text = desc  # chỉ dùng desc nếu không có title

    # ── NLP features gốc ──────────────────────────────────────────────────────
    df['feat_oto']        = text.str.contains('xe hơi|ô tô|oto').astype(int)
    df['feat_tranh']      = text.str.contains('tránh').astype(int)
    df['feat_no_hau']     = text.str.contains('nở hậu').astype(int)
    df['feat_thang_may']  = text.str.contains('thang máy').astype(int)
    df['feat_kinh_doanh'] = text.str.contains('kinh doanh|buôn bán').astype(int)
    df['feat_mat_tien']   = text.str.contains('mặt phố|mặt đường').astype(int)
    df['feat_noi_that']   = text.str.contains('nội thất|đầy đủ|tiện nghi').astype(int)
    df['feat_so_do']      = text.str.contains('sổ đỏ|sổ hồng').astype(int)
    df['feat_chinh_chu']  = text.str.contains('chính chủ').astype(int)

    # ── NLP features mới (V11) ─────────────────────────────────────────────────
    df['feat_view_nui']      = text.str.contains('view núi|view đồi').astype(int)
    df['feat_view_ho_song']  = text.str.contains('view hồ|view sông|sát hồ|ven hồ').astype(int)
    df['feat_view_canh_dong']= text.str.contains('view cánh đồng|cánh đồng').astype(int)
    df['feat_khuon_vien']    = text.str.contains('sẵn ao|vườn cây|cây ăn trái|nhà vườn').astype(int)
    df['feat_nghi_duong']    = text.str.contains('nghỉ dưỡng|homestay|farmstay|villa').astype(int)
    df['feat_nha_xuong']     = text.str.contains('nhà xưởng').astype(int)
    df['feat_phan_lo']       = text.str.contains('phân lô').astype(int)
    df['feat_f0']            = text.str.contains('f0|chưa qua đầu tư').astype(int)
    df['feat_san_nha']       = text.str.contains('sẵn nhà|nhà cấp 4|ở ngay').astype(int)
    df['feat_duong_nhua']    = text.str.contains('đường nhựa').astype(int)
    df['feat_duong_betong']  = text.str.contains('đường bê tông').astype(int)
    df['feat_truc_chinh']    = text.str.contains('trục chính|đường tỉnh|tỉnh lộ|quốc lộ|đường lớn').astype(int)
    df['feat_phap_ly_chuan'] = text.str.contains('sẵn sổ|sang tên luôn|pháp lý chuẩn').astype(int)
    df['feat_du_lich']       = text.str.contains('khu du lịch|resort').astype(int)
    df['feat_truong_hoc']    = text.str.contains('trường học').astype(int)
    df['feat_cho']           = text.str.contains('chợ').astype(int)
    df['feat_nga_ba_tu']     = text.str.contains('ngã 3|ngã 4|ngã tư').astype(int)
    # ── Bổ sung NLP features (V12) ─────────────────────────────────────────────
    df['feat_view_bien']     = text.str.contains('biển').astype(int)
    df['feat_goc']           = text.str.contains('góc').astype(int)
    df['feat_cong_vien']     = text.str.contains('công viên').astype(int)
    df['feat_sieu_thi']      = text.str.contains('siêu thị|mart').astype(int)
    df['feat_benh_vien']     = text.str.contains('bệnh viện|bv').astype(int)
    df['feat_tttm']          = text.str.contains('trung tâm thương mại|tttm').astype(int)

    # ── Khoảng cách địa lý ────────────────────────────────────────────────────
    df['dist_to_metro'] = df.apply(
        lambda r: min(np.sqrt((r['Tọa độ x'] - mx)**2 + (r['Tọa độ y'] - my)**2)
                      for mx, my in METRO_STATIONS), axis=1
    )
    df['dist_to_center'] = np.sqrt(
        (df['Tọa độ x'] - CENTER_LAT)**2 + (df['Tọa độ y'] - CENTER_LON)**2
    )

    # ── Cột tổng hợp ─────────────────────────────────────────────────────────
    df['type_dist'] = df['Loại BĐS'].astype(str) + '_' + df['Quận'].astype(str)

    for col in ['Số tầng', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Mặt tiền', 'Đường vào']:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.extract(r'(\d+\.?\d*)')[0], errors='coerce'
        )

    df['can_goc'] = df['Căn góc'].astype(str).str.lower().isin(
        ['có', 'yes', '1', 'true', 'căn góc']
    ).astype(int)

    df['dien_tich_per_tang'] = df['Diện tích'] / df['Số tầng'].replace(0, np.nan).fillna(1)
    df['mat_tien_x_tang']    = df['Mặt tiền'] * df['Số tầng']

    return df

df_final = extract_features(df_final)
print(f'Feature extraction done. Shape: {df_final.shape}')

## 6. Spatial Clustering & Train/Test Split

In [ ]:
ALL_FEATURES = [
    'Loại BĐS', 'Quận', 'Địa chỉ 1', 'Diện tích',
    'Tọa độ x', 'Tọa độ y', 'Số tầng', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh',
    'Mặt tiền', 'Đường vào', 'Hướng nhà', 'Hướng ban công', 'Pháp lý', 'Nội thất',
    'can_goc',
    # NLP gốc
    'feat_kinh_doanh', 'feat_mat_tien', 'feat_oto', 'feat_tranh', 'feat_no_hau',
    'feat_thang_may', 'feat_noi_that', 'feat_so_do', 'feat_chinh_chu',
    # NLP mới V11
    'feat_view_nui', 'feat_view_ho_song', 'feat_view_canh_dong', 'feat_khuon_vien',
    'feat_nghi_duong', 'feat_nha_xuong', 'feat_phan_lo', 'feat_f0', 'feat_san_nha',
    'feat_duong_nhua', 'feat_duong_betong', 'feat_truc_chinh', 'feat_phap_ly_chuan',
    'feat_du_lich', 'feat_truong_hoc', 'feat_cho', 'feat_nga_ba_tu',
    # NLP bổ sung V12
    'feat_view_bien', 'feat_goc', 'feat_cong_vien', 'feat_sieu_thi',
    'feat_benh_vien', 'feat_tttm',
    # Khoảng cách + cluster
    'dist_to_metro', 'dist_to_center', 'type_dist', 'loc_cluster',
    # Engineered
    'dien_tich_per_tang', 'mat_tien_x_tang',
]

CATEGORICAL = [
    'Loại BĐS', 'Quận', 'Địa chỉ 1',
    'Hướng nhà', 'Hướng ban công', 'Pháp lý', 'Nội thất',
    'type_dist', 'loc_cluster'
]

# KMeans spatial clustering
coords = df_final[['Tọa độ x', 'Tọa độ y']].copy()
coords['Tọa độ x'] = coords['Tọa độ x'].fillna(coords['Tọa độ x'].median())
coords['Tọa độ y'] = coords['Tọa độ y'].fillna(coords['Tọa độ y'].median())

kmeans = MiniBatchKMeans(n_clusters=400, random_state=42, n_init=3)
df_final['loc_cluster'] = kmeans.fit_predict(coords)

X = df_final[ALL_FEATURES]
y = np.log1p(df_final['Price'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

encoder = ce.TargetEncoder(cols=CATEGORICAL)
X_train_enc = encoder.fit_transform(X_train, y_train)
X_test_enc  = encoder.transform(X_test)

print(f'Train: {X_train_enc.shape}, Test: {X_test_enc.shape}')
print(f'Total features: {len(ALL_FEATURES)}')

## 7. Optuna Tuning — XGBoost

In [ ]:
N_TRIALS = 20  # Tăng lên nếu muốn tìm tham số tốt hơn (mỗi trial ~3-5 phút)

def objective_xgb(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 2000, 5000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.03),
        'max_depth':         trial.suggest_int('max_depth', 10, 16),
        'subsample':         trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'tree_method': 'hist',
        'verbosity': 0,
    }
    m = xgb.XGBRegressor(**params)
    m.fit(X_train_enc, y_train)
    mape = mean_absolute_percentage_error(np.expm1(y_test), np.expm1(m.predict(X_test_enc)))
    return mape

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ XGB best MAPE: {study_xgb.best_value*100:.2f}%')
print(f'Best params: {study_xgb.best_params}')

## 8. Optuna Tuning — LightGBM

In [ ]:
def objective_lgb(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 2000, 5000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.03),
        'num_leaves':       trial.suggest_int('num_leaves', 255, 1023),
        'subsample':        trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_samples':trial.suggest_int('min_child_samples', 5, 30),
        'device': 'cpu',  
        'verbose': -1,
    }
    m = lgb.LGBMRegressor(**params)
    m.fit(X_train_enc, y_train)
    mape = mean_absolute_percentage_error(np.expm1(y_test), np.expm1(m.predict(X_test_enc)))
    return mape

study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✅ LGB best MAPE: {study_lgb.best_value*100:.2f}%')
print(f'Best params: {study_lgb.best_params}')

## 9. Final Training & Ensemble Blending

In [ ]:
print('Training final XGBoost...')
m_xgb = xgb.XGBRegressor(**study_xgb.best_params)
m_xgb.fit(X_train_enc, y_train)

print('Training final LightGBM...')
m_lgb = lgb.LGBMRegressor(**study_lgb.best_params)
m_lgb.fit(X_train_enc, y_train)

# Grid search trọng số ensemble
p_xgb = np.expm1(m_xgb.predict(X_test_enc))
p_lgb = np.expm1(m_lgb.predict(X_test_enc))
y_true = np.expm1(y_test)

best_w, best_mape = 0.5, 999
for w in np.arange(0.0, 1.01, 0.05):
    mape = mean_absolute_percentage_error(y_true, w * p_xgb + (1 - w) * p_lgb)
    if mape < best_mape:
        best_mape, best_w = mape, w

y_pred_final = best_w * p_xgb + (1 - best_w) * p_lgb

print(f'\n🏆 FINAL V11 MAPE: {best_mape*100:.2f}%')
print(f'   XGB weight: {best_w:.2f} | LGB weight: {1-best_w:.2f}')

# MAPE by BĐS type
eval_df = X_test[['Loại BĐS']].copy()
eval_df['y_true'] = y_true.values
eval_df['y_pred'] = y_pred_final
mape_by_type = (
    eval_df.groupby('Loại BĐS')
    .apply(lambda x: mean_absolute_percentage_error(x['y_true'], x['y_pred']) * 100)
    .rename('MAPE (%)')
    .round(2)
)
print('\nMAPE by BĐS type:')
print(mape_by_type.to_string())

## 10. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

fi_xgb = pd.Series(m_xgb.feature_importances_, index=ALL_FEATURES).sort_values(ascending=True).tail(30)

fig, ax = plt.subplots(figsize=(9, 10))
fi_xgb.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('XGBoost — Top 30 Feature Importance (V11)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig(f'{LAKEHOUSE_DIR}/feature_importance_v11.png', dpi=150)
plt.show()

## 11. Save Models to Lakehouse

In [ ]:
joblib.dump(m_xgb,   f'{LAKEHOUSE_DIR}/master_xgb_v11.joblib')
joblib.dump(m_lgb,   f'{LAKEHOUSE_DIR}/master_lgb_v11.joblib')
joblib.dump(encoder, f'{LAKEHOUSE_DIR}/master_encoder_v11.joblib')
joblib.dump(kmeans,  f'{LAKEHOUSE_DIR}/master_kmeans_v11.joblib')

# Lưu metadata
meta = {
    'version': 'V11',
    'mape': f'{best_mape*100:.2f}%',
    'xgb_weight': best_w,
    'lgb_weight': 1 - best_w,
    'features': ALL_FEATURES,
    'xgb_params': study_xgb.best_params,
    'lgb_params': study_lgb.best_params,
    'n_train': len(X_train),
    'n_test': len(X_test),
}

import json
with open(f'{LAKEHOUSE_DIR}/model_meta_v11.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print('✅ Models saved to Lakehouse:')
print(f'   master_xgb_v11.joblib')
print(f'   master_lgb_v11.joblib')
print(f'   master_encoder_v11.joblib')
print(f'   master_kmeans_v11.joblib')
print(f'   model_meta_v11.json')
print(f'\n📊 Final MAPE: {best_mape*100:.2f}%')

## 12. Evaluate on External HaNoiData
Evaluate the trained V11 ensemble on the held-out HaNoiData (HaNoiXtestData.csv / HaNoiYtestData.csv)

In [ ]:
# ── Load models from Lakehouse ──────────────────────────────────────────────────
import json

XGB_PATH    = f'{LAKEHOUSE_DIR}/master_xgb_v11.joblib'
LGB_PATH    = f'{LAKEHOUSE_DIR}/master_lgb_v11.joblib'
ENC_PATH    = f'{LAKEHOUSE_DIR}/master_encoder_v11.joblib'
KMEANS_PATH = f'{LAKEHOUSE_DIR}/master_kmeans_v11.joblib'
META_PATH   = f'{LAKEHOUSE_DIR}/model_meta_v11.json'

print('Loading models from Lakehouse...')
m_xgb   = joblib.load(XGB_PATH)
m_lgb   = joblib.load(LGB_PATH)
encoder = joblib.load(ENC_PATH)
kmeans  = joblib.load(KMEANS_PATH)

with open(META_PATH, 'r', encoding='utf-8') as f:
    meta = json.load(f)
ALL_FEATURES = meta['features']
best_w       = meta['xgb_weight']
print(f'Loaded XGBoost, LightGBM, encoder, kmeans, and metadata ✅')
print(f'  Best XGB weight: {best_w:.2f}')

# ── Load external HaNoiData for evaluation ─────────────────────────────────────
print('Loading external HaNoiData for evaluation...')
try:
    X_ext = pd.read_csv(f'{LAKEHOUSE_DIR}/HaNoiXtestData.csv', encoding='utf-8')
    y_ext = pd.read_csv(f'{LAKEHOUSE_DIR}/HaNoiYtestData.csv')
except FileNotFoundError:
    X_ext = pd.read_csv('HaNoiXtestData.csv', encoding='utf-8')
    y_ext = pd.read_csv('HaNoiYtestData.csv')

print(f'Loaded {len(X_ext):,} external samples')

# ── Rename columns ───────────────────────────────────────────────────────────
col_idx = X_ext.columns.tolist()
rename_map = {}
if len(col_idx) > 1: rename_map[col_idx[1]] = 'Loại BĐS'
if len(col_idx) > 3: rename_map[col_idx[3]] = 'Quận'
if len(col_idx) > 6: rename_map[col_idx[6]] = 'Diện tích'
if len(col_idx) > 7: rename_map[col_idx[7]] = 'Số phòng ngủ'

X_ext = X_ext.rename(columns={**rename_map,
    'Phuong': 'Phường Xã Thị trấn', 'dia_chi_2': 'Địa chỉ 2',
    'Mo_ta': 'Mô tả', 'Toa_do_x': 'Tọa độ x', 'Toa_do_y': 'Tọa độ y',
    'so_tang': 'Số tầng', 'mat_tien': 'Mặt tiền', 'duong_vao': 'Đường vào',
    'tieu_de': 'Tiêu đề',
})

# ── Fill missing columns ─────────────────────────────────────────────────────
for c in ['Địa chỉ 1', 'Căn góc', 'Hướng nhà', 'Hướng ban công', 'Pháp lý', 'Nội thất', 
          'Số phòng tắm - vệ sinh', 'Phường Xã Thị trấn', 'Tiêu đề']:
    if c not in X_ext.columns:
        X_ext[c] = np.nan

for col in ['Diện tích', 'Tọa độ x', 'Tọa độ y']:
    X_ext[col] = pd.to_numeric(X_ext[col], errors='coerce').fillna(0)

# ── Extract features ─────────────────────────────────────────────────────────
X_ext = extract_features(X_ext.copy())

# ── Predict loc_cluster ──────────────────────────────────────────────────────
X_ext['loc_cluster'] = kmeans.predict(X_ext[['Tọa độ x', 'Tọa độ y']].fillna(
    X_ext[['Tọa độ x', 'Tọa độ y']].median()
))

# ── Encode & Predict ─────────────────────────────────────────────────────────
X_ext_enc = encoder.transform(X_ext[ALL_FEATURES])
p_xgb_ext = np.expm1(m_xgb.predict(X_ext_enc))
p_lgb_ext = np.expm1(m_lgb.predict(X_ext_enc))
y_pred_ext = best_w * p_xgb_ext + (1 - best_w) * p_lgb_ext

# ── True prices ──────────────────────────────────────────────────────────────
y_true_ext = y_ext['mean_unique_khoang_gia_million'] * 1e6

# ── Metrics ──────────────────────────────────────────────────────────────────
mape_ext = mean_absolute_percentage_error(y_true_ext, y_pred_ext)
print(f'\n{"═"*55}')
print(f'  📊 HaNoiData External Evaluation (V11)')
print(f'{"═"*55}')
print(f'  MAPE:  {mape_ext*100:.2f}%')
print(f'  Samples: {len(y_true_ext):,}')
print(f'  Best ensemble weights: XGB={best_w:.2f}, LGB={1-best_w:.2f}')
print(f'{"═"*55}')

# ── By district ──────────────────────────────────────────────────────────────
eval_ext = X_ext[['Quận']].copy()
eval_ext['y_true'] = y_true_ext.values
eval_ext['y_pred'] = y_pred_ext
mape_by_district = (
    eval_ext.groupby('Quận')
    .apply(lambda x: mean_absolute_percentage_error(x['y_true'], x['y_pred']) * 100)
    .rename('MAPE (%)')
    .round(2)
    .sort_values()
)
print('\nMAPE by District:')
print(mape_by_district.to_string())

# ── By type ──────────────────────────────────────────────────────────────────
if 'Loại BĐS' in X_ext.columns:
    eval_ext['Loại BĐS'] = X_ext['Loại BĐS'].values
    mape_by_type = (
        eval_ext.groupby('Loại BĐS')
        .apply(lambda x: mean_absolute_percentage_error(x['y_true'], x['y_pred']) * 100)
        .rename('MAPE (%)')
        .round(2)
    )
    print('\nMAPE by BĐS Type:')
    print(mape_by_type.to_string())

## 13. Check Overlap: Training Data vs HaNoiData
Kiểm tra xem X_train (từ ads_data_2026_03) có bị trùng với HaNoiData không — nếu có, kết quả evaluation sẽ bị bias.

In [ ]:
# ── Check overlap: Training data vs HaNoiData ──────────────────────────────────
print('=' * 60)
print('  🔍 Checking for overlap between Training Data & HaNoiData')
print('=' * 60)

# Dùng X_train và y_train đã có sẵn từ section 6
print(f'\nTraining data (X_train):  {len(X_train):,} rows')
print(f'HaNoiData (external):      {len(X_ext):,} rows')

# ── Prepare merge keys ────────────────────────────────────────────────────────
# Round price to millions, round coords to 4 decimal places (~11m precision)
train_tmp = X_train[[
    'Loại BĐS', 'Quận', 'Diện tích', 'Tọa độ x', 'Tọa độ y'
]].copy()
train_tmp['_idx'] = train_tmp.index  # keep original index to map back to df_final
train_tmp['_price_m'] = (np.expm1(y_train) / 1e6).round(0).values
train_tmp['_lat_r'] = train_tmp['Tọa độ x'].round(4)
train_tmp['_lon_r'] = train_tmp['Tọa độ y'].round(4)
train_tmp = train_tmp.dropna(subset=['_lat_r', '_lon_r'])

hanoi_tmp = X_ext[[
    'Loại BĐS', 'Quận', 'Diện tích', 'Tọa độ x', 'Tọa độ y'
]].copy()
hanoi_tmp['_hanoi_idx'] = hanoi_tmp.index  # keep original index to map back to X_ext
hanoi_tmp['_price_m'] = (y_true_ext / 1e6).round(0)
hanoi_tmp['_lat_r'] = pd.to_numeric(hanoi_tmp['Tọa độ x'], errors='coerce').round(4)
hanoi_tmp['_lon_r'] = pd.to_numeric(hanoi_tmp['Tọa độ y'], errors='coerce').round(4)
hanoi_tmp = hanoi_tmp.dropna(subset=['_lat_r', '_lon_r'])

# ── Check 1: Match on (Loại BĐS, Quận, Diện tích, price) ─────────────────────
exact_overlap = pd.merge(
    train_tmp, hanoi_tmp,
    on=['Loại BĐS', 'Quận', 'Diện tích', '_price_m'],
    how='inner'
)
print(f'\n[1] Overlap (BĐS type, District, Area, rounded price):')
print(f'    {len(exact_overlap):,} rows  ({len(exact_overlap)/len(hanoi_tmp)*100:.2f}% of HaNoiData)')

# ── Check 2: Match on (Loại BĐS, Quận, Diện tích, price, coords) ─────────────
coord_overlap = pd.merge(
    train_tmp, hanoi_tmp,
    on=['Loại BĐS', 'Quận', 'Diện tích', '_price_m', '_lat_r', '_lon_r'],
    how='inner'
)
print(f'\n[2] Overlap (BĐS type, District, Area, price, coordinates):')
print(f'    {len(coord_overlap):,} rows  ({len(coord_overlap)/len(hanoi_tmp)*100:.2f}% of HaNoiData)')

# ── Print duplicate rows interleaved (BOTH sides) ────────────────────────────
if len(coord_overlap) > 0:
    OVERLAP_COLS = [
        'Loại quảng cáo', 'Loại BĐS', 'Tỉnh thành phố', 'Quận', 'Địa chỉ 1', 'Căn góc',
        'Diện tích', 'Số phòng ngủ', 'Số phòng tắm - vệ sinh', 'Tên dự án',
        'Hướng nhà', 'Hướng ban công', 'Số tầng', 'Mặt tiền', 'Đường vào',
    ]
    
    # ── Build interleaved rows: X_train then HaNoiData for each match pair ──
    interleaved_rows = []
    for _, pair in coord_overlap.iterrows():
        # X_train row
        train_row = df_final.loc[pair['_idx'], OVERLAP_COLS].to_dict()
        train_row['Source'] = 'X_train'
        interleaved_rows.append(train_row)
        # HaNoiData row
        hanoi_row = {}
        for c in OVERLAP_COLS:
            hanoi_row[c] = X_ext.loc[pair['_hanoi_idx'], c] if c in X_ext.columns else np.nan
        hanoi_row['Source'] = 'HaNoiData'
        interleaved_rows.append(hanoi_row)
    
    interleaved = pd.DataFrame(interleaved_rows)[['Source'] + OVERLAP_COLS]
    print(f'\n📋 Duplicate rows (interleaved X_train ⇔ HaNoiData comparator):')
    print(f'    Total match pairs: {len(coord_overlap):,}')
    with pd.option_context('display.max_columns', None, 'display.width', 300, 'display.max_colwidth', 25):
        display(interleaved)

# ── Check 3: KDTree nearest-neighbor search ────────────────────────────────────
from sklearn.neighbors import KDTree
if len(train_tmp) > 0 and len(hanoi_tmp) > 0:
    train_coords = train_tmp[['_lat_r', '_lon_r']].values
    hanoi_coords = hanoi_tmp[['_lat_r', '_lon_r']].values
    tree = KDTree(train_coords)
    distances, _ = tree.query(hanoi_coords, k=1)
    close_neighbors = int((distances.flatten() < 0.001).sum())
    print(f'\n[3] HaNoiData rows with nearby training row (coord dist < 0.001):')
    print(f'    {close_neighbors:,} rows  ({close_neighbors/len(hanoi_tmp)*100:.2f}% of HaNoiData)')

# ── Summary ────────────────────────────────────────────────────────────────────
print(f'\n{"─"*55}')
if len(coord_overlap) > 0:
    print(f'  ⚠️  WARNING: Found {len(coord_overlap):,} exact duplicate(s)')
    print(f'     between training data and HaNoiData!')
    print(f'     Model MAPE on HaNoiData may be OVERESTIMATED (biased) ❌')
elif len(exact_overlap) > 0:
    print(f'  ⚠️  Found {len(exact_overlap):,} potential overlap(s) by area/price,')
    print(f'     but coordinates differ — likely different listings, safe ✅')
else:
    print(f'  ✅ No duplicates found between training data and HaNoiData')
    print(f'     HaNoiData is a clean held-out evaluation set ✅')
print(f'{"─"*55}')